# ABC2Vec — Notebook 12: Ablation Study

Measure the contribution of each pre-training objective by training models with
different objective combinations.

**Ablation configurations:**

| Config | MMM | SCL | TI | VAC | Description |
|--------|-----|-----|-----|-----|-------------|
| Full   | ✓   | ✓   | ✓   | ✓   | All objectives (default) |
| −SCL   | ✓   |     | ✓   | ✓   | Remove Section Contrastive |
| −TI    | ✓   | ✓   |     | ✓   | Remove Transposition Invariance |
| −VAC   | ✓   | ✓   | ✓   |     | Remove Variant-Aware Contrastive |
| MMM-only | ✓ |     |     |     | Baseline: only masked modeling |
| SCL+TI |     | ✓   | ✓   |     | Contrastive only (no MLM) |
| Random |     |     |     |     | Random untrained baseline |

Each is evaluated on retrieval (MRR, R@5) and classification (accuracy).

In [ ]:
import os, sys, json, copy, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

try:
    import faiss
except ImportError:
    !pip install faiss-cpu
    import faiss

PROJECT_DIR = Path('/Volumes/LLModels/ABC2VEC')
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
BENCHMARK_DIR = PROJECT_DIR / 'data' / 'benchmark'
RESULTS_DIR = PROJECT_DIR / 'results'
ABLATION_DIR = PROJECT_DIR / 'checkpoints' / 'ablation'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR))

device = torch.device('cuda' if torch.cuda.is_available() else 
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

## 12.1 Define Ablation Configurations

In [ ]:
from abc2vec.model import ABC2VecModel, ABC2VecConfig
from abc2vec.tokenizer import ABCVocabulary, BarPatchifier, ABC2VecDataset, SectionPairDataset
from abc2vec.losses import (
    MaskedMusicModelingLoss, SectionContrastiveLoss,
    TranspositionInvarianceLoss, VariantAwareContrastiveLoss
)

# Load vocab
vocab = ABCVocabulary()
vocab_path = PROJECT_DIR / 'data' / 'processed' / 'vocabulary.json'
if vocab_path.exists():
    vocab.load(vocab_path)

patchifier = BarPatchifier(vocab, max_bar_length=64, max_bars=64)

# Ablation configs: {name: {lambda_mmm, lambda_scl, lambda_ti, lambda_vac}}
ABLATION_CONFIGS = {
    'Full (MMM+SCL+TI+VAC)': {'lambda_mmm': 1.0, 'lambda_scl': 0.5, 'lambda_ti': 0.3, 'lambda_vac': 0.5},
    '-SCL (MMM+TI+VAC)':     {'lambda_mmm': 1.0, 'lambda_scl': 0.0, 'lambda_ti': 0.3, 'lambda_vac': 0.5},
    '-TI (MMM+SCL+VAC)':     {'lambda_mmm': 1.0, 'lambda_scl': 0.5, 'lambda_ti': 0.0, 'lambda_vac': 0.5},
    '-VAC (MMM+SCL+TI)':     {'lambda_mmm': 1.0, 'lambda_scl': 0.5, 'lambda_ti': 0.3, 'lambda_vac': 0.0},
    'MMM only':              {'lambda_mmm': 1.0, 'lambda_scl': 0.0, 'lambda_ti': 0.0, 'lambda_vac': 0.0},
    'SCL+TI only':           {'lambda_mmm': 0.0, 'lambda_scl': 0.5, 'lambda_ti': 0.3, 'lambda_vac': 0.0},
}

print("Ablation configurations:")
for name, lambdas in ABLATION_CONFIGS.items():
    active = [k.split('_')[1].upper() for k, v in lambdas.items() if v > 0]
    print(f"  {name:30s}  active: {', '.join(active) if active else 'none'}")

## 12.2 Ablation Training Loop

In [ ]:
# Load training data
train_df = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
val_df = pd.read_parquet(PROCESSED_DIR / 'val.parquet')

print(f"Train: {len(train_df)}, Val: {len(val_df)}")

# Shorter training for ablation (5 epochs instead of 20)
ABLATION_EPOCHS = 5
ABLATION_BATCH_SIZE = 32
ABLATION_LR = 1e-4
ABLATION_GRAD_ACCUM = 2  # Smaller for speed

config = ABC2VecConfig(
    vocab_size=len(vocab),
    max_bar_len=64, max_bars=64,
    d_model=256, n_heads=8, n_layers=6,
    d_ff=1024, d_embed=128, dropout=0.1,
)

In [ ]:
def train_ablation(config_name: str, lambdas: dict, 
                   train_df=train_df, val_df=val_df,
                   n_epochs=ABLATION_EPOCHS) -> dict:
    """
    Train one ablation variant and return final metrics.
    """
    print(f"\n{'='*60}")
    print(f"Ablation: {config_name}")
    print(f"{'='*60}")
    
    ckpt_path = ABLATION_DIR / f"{config_name.replace(' ', '_').replace('(', '').replace(')', '')}.pt"
    
    # Check if already trained
    if ckpt_path.exists():
        print(f"  Loading from cache: {ckpt_path.name}")
        checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
        model = ABC2VecModel(config).to(device)
        model.load_state_dict(checkpoint['model_state_dict'])
        return {'model': model, 'losses': checkpoint.get('losses', [])}
    
    # Create model
    model = ABC2VecModel(config).to(device)
    
    # Loss functions
    mmm_loss_fn = MaskedMusicModelingLoss() if lambdas['lambda_mmm'] > 0 else None
    scl_loss_fn = SectionContrastiveLoss() if lambdas['lambda_scl'] > 0 else None
    ti_loss_fn = TranspositionInvarianceLoss() if lambdas['lambda_ti'] > 0 else None
    vac_loss_fn = VariantAwareContrastiveLoss() if lambdas['lambda_vac'] > 0 else None
    
    # Dataset & DataLoader
    dataset = ABC2VecDataset(
        train_df['abc_body'].tolist(), patchifier,
        augment_transpose=lambdas['lambda_ti'] > 0
    )
    loader = DataLoader(dataset, batch_size=ABLATION_BATCH_SIZE, 
                        shuffle=True, num_workers=0, drop_last=True)
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=ABLATION_LR, weight_decay=0.01)
    
    losses_history = []
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{n_epochs}")
        for batch_idx, batch in enumerate(pbar):
            bar_ids = batch['bar_ids'].to(device)
            bar_mask = batch['bar_mask'].to(device)
            
            total_loss = torch.tensor(0.0, device=device)
            
            # MMM
            if mmm_loss_fn is not None:
                masked_ids, mask_positions = model.mask_bars(bar_ids, bar_mask, mask_prob=0.15)
                mmm_logits = model.forward_mmm(masked_ids, bar_mask, mask_positions)
                mmm_targets = bar_ids[mask_positions]  # Original characters
                loss_mmm = mmm_loss_fn(mmm_logits, mmm_targets)
                total_loss = total_loss + lambdas['lambda_mmm'] * loss_mmm
            
            # TI (transposition invariance)
            if ti_loss_fn is not None and 'bar_ids_transposed' in batch:
                bar_ids_t = batch['bar_ids_transposed'].to(device)
                bar_mask_t = batch['bar_mask_transposed'].to(device)
                emb_orig = model.get_embedding(bar_ids, bar_mask)
                emb_trans = model.get_embedding(bar_ids_t, bar_mask_t)
                loss_ti = ti_loss_fn(emb_orig, emb_trans)
                total_loss = total_loss + lambdas['lambda_ti'] * loss_ti
            
            # SCL (section contrastive) — simplified: use augmented pairs
            if scl_loss_fn is not None:
                # Use first/second half of batch as "section pairs"
                half = bar_ids.size(0) // 2
                if half > 1:
                    emb_a = model.get_embedding(bar_ids[:half], bar_mask[:half])
                    emb_b = model.get_embedding(bar_ids[half:2*half], bar_mask[half:2*half])
                    loss_scl = scl_loss_fn(emb_a, emb_b)
                    total_loss = total_loss + lambdas['lambda_scl'] * loss_scl
            
            if total_loss.requires_grad:
                total_loss = total_loss / ABLATION_GRAD_ACCUM
                total_loss.backward()
                
                if (batch_idx + 1) % ABLATION_GRAD_ACCUM == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    optimizer.zero_grad()
            
            epoch_loss += total_loss.item() * ABLATION_GRAD_ACCUM
            n_batches += 1
            pbar.set_postfix({'loss': epoch_loss / n_batches})
        
        avg_loss = epoch_loss / max(n_batches, 1)
        losses_history.append(avg_loss)
        print(f"  Epoch {epoch+1}: loss = {avg_loss:.4f}")
    
    # Save checkpoint
    torch.save({
        'model_state_dict': model.state_dict(),
        'config_name': config_name,
        'lambdas': lambdas,
        'losses': losses_history,
    }, ckpt_path)
    print(f"  Saved to {ckpt_path.name}")
    
    return {'model': model, 'losses': losses_history}

In [ ]:
# Train all ablation variants
ablation_models = {}

for config_name, lambdas in ABLATION_CONFIGS.items():
    result = train_ablation(config_name, lambdas)
    ablation_models[config_name] = result

# Add random baseline
random_model = ABC2VecModel(config).to(device)
random_model.eval()
ablation_models['Random (untrained)'] = {'model': random_model, 'losses': []}

print(f"\nTrained {len(ablation_models)} models.")

## 12.3 Evaluate All Variants

In [ ]:
# Load benchmark data
queries_df = pd.read_parquet(BENCHMARK_DIR / 'retrieval_queries.parquet')
corpus_df = pd.read_parquet(BENCHMARK_DIR / 'retrieval_corpus.parquet')
class_df = pd.read_parquet(BENCHMARK_DIR / 'classification_test.parquet')

print(f"Retrieval: {len(queries_df)} queries, {len(corpus_df)} corpus")
print(f"Classification: {len(class_df)} tunes")

@torch.no_grad()
def encode_tunes(abc_texts, model, patchifier, device, batch_size=64):
    model.eval()
    all_emb = []
    for i in range(0, len(abc_texts), batch_size):
        batch = abc_texts[i:i+batch_size]
        bar_ids_list, bar_mask_list = [], []
        for text in batch:
            bars = patchifier.patchify(text)
            ids, mask = patchifier.encode(bars)
            bar_ids_list.append(ids)
            bar_mask_list.append(mask)
        bar_ids = torch.stack(bar_ids_list).to(device)
        bar_mask = torch.stack(bar_mask_list).to(device)
        emb = model.get_embedding(bar_ids, bar_mask)
        all_emb.append(emb.cpu().numpy())
    return np.concatenate(all_emb, axis=0)

def eval_retrieval(query_emb, corpus_emb, query_families, corpus_families, k=5):
    """Quick retrieval evaluation: MRR and Recall@K."""
    d = query_emb.shape[1]
    
    # Normalize
    qn = query_emb / np.maximum(np.linalg.norm(query_emb, axis=1, keepdims=True), 1e-8)
    cn = corpus_emb / np.maximum(np.linalg.norm(corpus_emb, axis=1, keepdims=True), 1e-8)
    
    index = faiss.IndexFlatIP(d)
    index.add(cn.astype(np.float32))
    _, indices = index.search(qn.astype(np.float32), k)
    
    mrr, recall_at_k = 0.0, 0.0
    n_valid = 0
    for i in range(len(query_families)):
        retrieved = corpus_families[indices[i]]
        is_rel = (retrieved == query_families[i])
        n_relevant = np.sum(corpus_families == query_families[i])
        if n_relevant == 0:
            continue
        n_valid += 1
        positions = np.where(is_rel)[0]
        if len(positions) > 0:
            mrr += 1.0 / (positions[0] + 1)
        recall_at_k += np.sum(is_rel) / min(n_relevant, k)
    
    return {
        'MRR': mrr / max(n_valid, 1),
        'Recall@5': recall_at_k / max(n_valid, 1),
    }

def eval_classification(emb, labels):
    """Quick classification with linear probe."""
    le = LabelEncoder()
    y = le.fit_transform(labels)
    clf = LogisticRegression(max_iter=2000, C=1.0, multi_class='multinomial', solver='lbfgs')
    scores = cross_val_score(clf, emb, y, cv=5, scoring='accuracy')
    return {'Accuracy': scores.mean()}

In [ ]:
# Evaluate all ablation models
ablation_results = []

for config_name, result in tqdm(ablation_models.items(), desc="Evaluating"):
    model = result['model']
    model.eval()
    
    print(f"\nEvaluating: {config_name}")
    
    # Encode
    q_emb = encode_tunes(queries_df['abc_body'].tolist(), model, patchifier, device)
    c_emb = encode_tunes(corpus_df['abc_body'].tolist(), model, patchifier, device)
    cl_emb = encode_tunes(class_df['abc_body'].tolist(), model, patchifier, device)
    
    # Retrieval
    ret_metrics = eval_retrieval(
        q_emb, c_emb,
        queries_df['family_id'].values, corpus_df['family_id'].values
    )
    
    # Classification
    cls_metrics = eval_classification(cl_emb, class_df['tune_type'].values)
    
    row = {'Config': config_name}
    row.update(ret_metrics)
    row.update(cls_metrics)
    ablation_results.append(row)
    
    print(f"  MRR={ret_metrics['MRR']:.4f}, R@5={ret_metrics['Recall@5']:.4f}, "
          f"Acc={cls_metrics['Accuracy']:.4f}")

## 12.4 Results Table and Visualization

In [ ]:
ablation_df = pd.DataFrame(ablation_results)
ablation_df = ablation_df.sort_values('MRR', ascending=False)

print("\n" + "=" * 70)
print("Ablation Study Results")
print("=" * 70)
print(ablation_df.to_string(index=False, float_format='%.4f'))

ablation_df.to_csv(RESULTS_DIR / 'ablation_results.csv', index=False)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

configs = ablation_df['Config'].tolist()
colors = plt.cm.Set2(np.linspace(0, 1, len(configs)))

# MRR
axes[0].barh(configs, ablation_df['MRR'], color=colors)
axes[0].set_xlabel('MRR')
axes[0].set_title('Retrieval: MRR')
axes[0].set_xlim(0, 1)

# Recall@5
axes[1].barh(configs, ablation_df['Recall@5'], color=colors)
axes[1].set_xlabel('Recall@5')
axes[1].set_title('Retrieval: Recall@5')
axes[1].set_xlim(0, 1)

# Classification accuracy
axes[2].barh(configs, ablation_df['Accuracy'], color=colors)
axes[2].set_xlabel('Accuracy')
axes[2].set_title('Tune Type Classification')
axes[2].set_xlim(0, 1)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ablation_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 12.5 Training Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for config_name, result in ablation_models.items():
    if result['losses']:
        ax.plot(range(1, len(result['losses'])+1), result['losses'],
                'o-', label=config_name, linewidth=2)

ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Ablation Training Curves')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ablation_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 12.6 Contribution Analysis

In [ ]:
# Calculate marginal contribution of each objective
full_row = ablation_df[ablation_df['Config'].str.startswith('Full')].iloc[0]

print("Marginal Contribution of Each Objective:")
print("(Measured as drop when removed from full model)")
print()

for metric in ['MRR', 'Recall@5', 'Accuracy']:
    print(f"\n{metric}:")
    full_val = full_row[metric]
    
    for _, row in ablation_df.iterrows():
        if row['Config'] == full_row['Config']:
            continue
        delta = full_val - row[metric]
        direction = '↓' if delta > 0 else '↑'
        print(f"  {row['Config']:30s}: {row[metric]:.4f}  ({direction}{abs(delta):.4f})")

# Save contribution analysis
print("\n\nKey takeaway: The objective with the largest ↓ when removed")
print("contributes the most to that metric.")